# CloudScalerEnv Final-Round Playbook

This notebook is a practical execution guide for your current CloudScalerEnv project.
It combines:
- reproducible setup
- local OpenEnv compliance checks
- baseline scoring and seed-sweep evidence
- API smoke tests
- final-round readiness checklist

Use this notebook as your single source of truth before submissions and review calls.

## 1. Why This Notebook

This notebook is structured for fast, repeatable validation before submission:
- one place for config + run flow
- multi-run evidence instead of a single lucky run
- verification-first reporting (prove behavior, not just claim it)
- clear outputs that reviewers can inspect quickly

It uses the same practical pattern end to end:
- deterministic grader with stochastic trajectories
- explicit validation output
- repeatable commands for score and API checks

In [ ]:
from pathlib import Path
import subprocess
import json
import re
import time

PROJECT_ROOT = Path.cwd()
print('Project root:', PROJECT_ROOT)

required = [
    'openenv.yaml',
    'validate_local.py',
    'src/baseline.py',
    'src/app.py',
]

missing = [p for p in required if not (PROJECT_ROOT / p).exists()]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

print('All required project files are present.')

In [ ]:
def run_cmd(cmd):
    print('\n$ ' + cmd)
    completed = subprocess.run(
        cmd,
        shell=True,
        cwd=str(PROJECT_ROOT),
        text=True,
        capture_output=True
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    print('Exit code:', completed.returncode)
    return completed

# Optional: ensure dependencies are installed
# run_cmd('python -m pip install -r requirements.txt')
print('Helper ready. Uncomment install line if needed.')

## 2. Run OpenEnv Local Validator

Expected: `Result: 5/5 checks passed`.

This validates:
- metadata contract
- typed models
- environment interface
- episode lifecycle
- task + grader behavior

In [ ]:
validator = run_cmd('python validate_local.py')
validator_ok = 'Result: 5/5 checks passed' in validator.stdout
print('Validator passed:', validator_ok)

## 3. Baseline Benchmark (Default Task Seeds)

This should run the heuristic baseline over easy, medium, and hard tasks.

Current tuned target from project work:
- stronger medium/hard performance
- mean around the 0.70 zone

In [ ]:
baseline = run_cmd('python -m src.baseline --mode heuristic')

scores = {}
for line in baseline.stdout.splitlines():
    m = re.search(r'Task: (\w+) .* Grader Score 0.0-1.0: ([0-9.]+)', line)
    if m:
        scores[m.group(1)] = float(m.group(2))

m_mean = re.search(r'Overall Mean Score: ([0-9.]+)', baseline.stdout)
mean_score = float(m_mean.group(1)) if m_mean else None

print('Parsed scores:', scores)
print('Parsed mean:', mean_score)

## 4. Seed Sweep Evidence (Non-Constant Scoring)

A key review criterion is proving your grader is not constant-return across stochastic trajectories.

Run a sweep and keep this output for defense.

In [ ]:
seed_sweep = run_cmd('python -m src.baseline --mode heuristic --seed-sweep 11,22,33,44,55')
print('Seed sweep complete.')

## 5. API Smoke Test (Health, Reset, Step, State)

This launches the FastAPI app temporarily, verifies core OpenEnv endpoints, then stops the server.

In [ ]:
import httpx

server = subprocess.Popen(
    'python -m src.app',
    cwd=str(PROJECT_ROOT),
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

try:
    time.sleep(2.5)
    base = 'http://127.0.0.1:7860'

    with httpx.Client(timeout=10.0) as client:
        health = client.get(f'{base}/health')
        reset = client.post(f'{base}/reset', json={'task_id': 'easy-memory-leak'})
        step = client.post(f'{base}/step', json={'action_type': 'do_nothing'})
        state = client.get(f'{base}/state')

    print('health:', health.status_code, health.json())
    print('reset:', reset.status_code)
    print('step:', step.status_code)
    print('state:', state.status_code)

finally:
    server.terminate()
    try:
        server.wait(timeout=5)
    except Exception:
        server.kill()
    print('Server stopped.')

## 6. Final-Round Readiness Checklist

Mark this before upload/review:
- [ ] `validate_local.py` is 5/5 pass
- [ ] baseline run is captured with task scores + mean
- [ ] seed-sweep output is captured (proof of variability)
- [ ] `/health`, `/reset`, `/step`, `/state` all return 200
- [ ] README and defense guide numbers match latest run
- [ ] no hardcoded secrets in notebook/config/docs
- [ ] HF Space is updated with final frozen code

## 7. High-Impact Improvements (If You Still Have Time)

1. Add automated regression tests for score drift
- Save expected score ranges for default seeds and fail CI if outside band.

2. Add verifier-style checks to your own benchmark loop
- Example: hard task must keep crash_count <= threshold and uptime >= threshold.

3. Expand baseline analysis output
- Track avg crashes, avg SLA violations, avg budget use per task in one table.

4. Add one ablation report
- Show impact of policy-only vs grader-only tuning to demonstrate intentional design.

5. Keep a short defense script ready
- 30-second pitch
- 2-minute architecture + grading explanation
- 3 evidence artifacts: validator, baseline, seed sweep

In [ ]:
# Optional helper to collect concise evidence in one dict
evidence = {
    'validator_passed': bool(globals().get('validator_ok', False)),
    'baseline_scores': globals().get('scores', {}),
    'baseline_mean': globals().get('mean_score', None),
}
print(json.dumps(evidence, indent=2))

## 8. Submission Readiness Check

> Run this section right before resubmission.

Checklist:
- `inference.py` exists in project root
- Uses `OpenAI` client for LLM calls
- Reads env vars: `API_BASE_URL` (default), `MODEL_NAME` (default), `HF_TOKEN` (required)
- Emits strict line formats: `[START]`, `[STEP]`, `[END]`
- `[END]` always emitted after episode cleanup
- Space is running and not stuck in building state
- Runtime fits resource limits (2 vCPU, 8 GB RAM)

In [ ]:
import re
from pathlib import Path

inf_path = PROJECT_ROOT / 'inference.py'
content = inf_path.read_text(encoding='utf-8')

checks = {
    'inference_in_root': inf_path.exists(),
    'uses_openai_client': 'from openai import OpenAI' in content or 'OpenAI(' in content,
    'has_api_base_default': bool(re.search(r'API_BASE_URL\s*=\s*os\.getenv\(\s*"API_BASE_URL"\s*,', content)),
    'has_model_name_default': bool(re.search(r'MODEL_NAME\s*=\s*os\.getenv\(\s*"MODEL_NAME"\s*,', content)),
    'requires_hf_token': 'HF_TOKEN environment variable is required' in content,
    'has_start_tag': '[START]' in content,
    'has_step_tag': '[STEP]' in content,
    'has_end_tag': '[END]' in content,
    'end_has_only_required_fields': bool(re.search(r'\[END\]\s+success=\{?success_str\}?\s+steps=\{?steps\}?\s+rewards=', content)),
}

print('Inference compliance checks:')
for k, v in checks.items():
    print(f'- {k}: {v}')

all_ok = all(checks.values())
print('\nOverall pass:', all_ok)

print('\nQuick runtime test (without real token) expects ValueError:')
result = run_cmd('python inference.py')
print('HF_TOKEN required check triggered:', 'HF_TOKEN environment variable is required' in (result.stderr + result.stdout))